In [20]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv("/content/GNSS_LowRain_Dataset.csv")

In [22]:
print(df.shape)
print(df.info())

(2000, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time(ms)     2000 non-null   int64  
 1   Temp(C)      2000 non-null   float64
 2   Humidity(%)  2000 non-null   float64
 3   RainAnalog   2000 non-null   int64  
 4   RainDigital  2000 non-null   int64  
 5   SNR1         2000 non-null   int64  
 6   SNR2         2000 non-null   int64  
 7   SNR3         2000 non-null   int64  
 8   SNR4         2000 non-null   int64  
 9   SNR5         2000 non-null   int64  
 10  SNR6         2000 non-null   int64  
 11  SNR7         2000 non-null   int64  
 12  SNR8         2000 non-null   int64  
 13  SNR9         2000 non-null   int64  
 14  SNR10        2000 non-null   int64  
 15  SNR11        2000 non-null   int64  
 16  SNR12        2000 non-null   int64  
 17  SNR13        2000 non-null   int64  
 18  SNR14        2000 non-null   int64  


In [23]:
(df == 0).sum()

,0
Time(ms),1
Temp(C),0
Humidity(%),0
RainAnalog,0
RainDigital,0
SNR1,0
SNR2,0
SNR3,0
SNR4,0
SNR5,0


In [24]:
df['RainDigital'] = 1

In [25]:
df['RainDigital'] = np.where(df['RainAnalog'] > 20, 1, 0)

In [26]:
snr_cols = [col for col in df.columns if "SNR" in col]

for col in snr_cols:
    df.loc[df[col] > 100, col] = np.nan
    df.loc[df[col] < 10, col] = np.nan

In [27]:
for col in snr_cols:
    df.loc[df[col] == 0, col] = np.nan

In [28]:
df.loc[df['Temp(C)'] == 0, 'Temp(C)'] = np.nan
df.loc[df['Humidity(%)'] == 0, 'Humidity(%)'] = np.nan

In [29]:
df = df.sort_values("Time(ms)")

df[snr_cols] = df[snr_cols].interpolate(method='linear')
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [30]:
df.fillna(method='bfill', inplace=True)
df.fillna(method='ffill', inplace=True)

/tmp/ipython-input-2053843447.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/tmp/ipython-input-2053843447.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [31]:
for col in snr_cols:
    df[col] = df[col].rolling(window=3, min_periods=1).mean()

In [32]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [33]:
df[snr_cols] = df[snr_cols].round().astype("Int64")

In [34]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)

In [35]:
df['SNR_Var'] = df[snr_cols].var(axis=1)

In [36]:
df['SNR_Diff'] = df['Mean_SNR'].diff().fillna(0)

In [37]:
df['Sat_Count'] = df[snr_cols].notna().sum(axis=1)

In [38]:
df.to_csv("Low_Rain_Clean.csv", index=False)